# Software profesional en Acústica 2024-25 (M2i)

*This notebook was adapted from Chapter 1 of [The FEniCS Tutorial Volume I](https://fenicsproject.org/pub/tutorial/sphinx1/) by Hans Petter Langtangen and Anders Logg, released under CC Attribution 4.0 license. It has been created by Xiangmin Jiao (University of Stony Brook University) and it is available in the repository [Unifem/FEniCS-note](https://github.com/unifem/fenics-notes). The FEniCS implementation has been replaced with NGSolve.*

This Jupyter Notebook has been designed to be run in [Google Colab](https://colab.research.google.com/). With this purpose the first cell install [NGSolve](https://ngsolve.org/) related packages in a clean machine (if they have not been previously installed). Typically, this installation takes less than 2 minutes.

In [17]:
try:
    import ngsolve
except ImportError:
    !wget "https://fem-on-colab.github.io/releases/ngsolve-install-release-complex.sh" -O "/tmp/ngsolve-install.sh" && bash "/tmp/ngsolve-install.sh"
    import ngsolve

# The equations of vibrations in fluid-structure interaction (displacement formulation)

Vibrations in linear elasticity is the study of how solid objects are responding to a mechanical vibration and become 
internally stressed due to prescribed time-harmonic loading conditions. It is an important problem
in modern engineering. Its corresponding PDE is a generalization of the
Helmholtz equation, and it is among one of the most popular PDEs in 
engineering. We now study its variational formulation of a fluid-structure interaction and how to solve
this problem using NGSolve in 2D.

## PDE problem

The time-harmonic equation governing vibrations of a fluid-structure problem involving a elastic solid structure $\Omega_S$ and a fluid domain $\Omega_F$ can be written as
\begin{align}
\tag{1}
&-\omega^2\rho_S\boldsymbol{u}_S -\boldsymbol{\nabla}\cdot\boldsymbol{\sigma} = \boldsymbol{0}\hbox{ in }\Omega_S,\\
&-\omega^2\rho_F\boldsymbol{u}_F -\rho_Fc^2\nabla\mathrm{div}\boldsymbol{u}_F = \boldsymbol{0}\hbox{ in }\Omega_F,
\tag{2}
\end{align}

where $\boldsymbol{\sigma}$ is the *stress tensor*, $c$ is the sound speed in the fluid domain, $\boldsymbol{u}_S$ and $\boldsymbol{u}_F$ are the displacement fields in the solid anf fluid domain,
$\rho_S$ and $\rho_F$ are the *mass density* of the solid and fluid domain, and $\omega$ the angular frequency. For isotropic materials, the stress tensor is further related to the deformation by 
the following two equations:
\begin{align}
\boldsymbol{\sigma} &= \lambda\,\hbox{tr}\,(\boldsymbol{\varepsilon}) \boldsymbol{I} + 2\mu\boldsymbol{\varepsilon},
\tag{3}\\
\boldsymbol{\varepsilon} &= \frac{1}{2}\left(\boldsymbol{\nabla} \boldsymbol{u}_S + (\boldsymbol{\nabla} \boldsymbol{u}_S)^{\top}\right),
\tag{4}
\end{align}
where $\boldsymbol{\varepsilon}$ is the *symmetric strain-rate tensor* (symmetric gradient), 
and $\boldsymbol{u}$ is the *displacement vector field*, $\boldsymbol{I}$ denotes the *identity tensor*, 
$\mathrm{tr}$ denotes the *trace operator* on a tensor, and $\lambda$ and $\mu$ 
are material properties known as *Lamé's elasticity parameters*.

We can combine (3) and (4) to obtain
\begin{equation}
\tag{5}
\boldsymbol{\boldsymbol{\sigma}} = \lambda(\boldsymbol{\nabla}\cdot \boldsymbol{u}_S)\boldsymbol{I} + \mu(\boldsymbol{\nabla} \boldsymbol{u}_S + (\boldsymbol{\nabla} \boldsymbol{u}_S)^{\top})
\end{equation}

Note that
(3)-(5)
can easily be transformed to a single vector PDE for $\boldsymbol{u}_S$, which is the
governing PDE for the unknown $\boldsymbol{u}_S$ (Navier's equation).  In the
derivation of the variational formulation, however, it is convenient
to keep the equations split as above.

## Variational formulation

The variational formulation of (1)-(4)
consists of forming the inner product of (1)-(2) and two *vector* test functions
$(\boldsymbol{v}_F,\boldsymbol{v}_S)\in \hat{V}$, where $\hat{V}$ is a vector-valued test function space in $\Omega_F$ and $\Omega_S$ such that 
\begin{align}
\boldsymbol{v}_F\cdot \mathbf{n} = \boldsymbol{v}_S\cdot \mathbf{n},\\
\boldsymbol{\sigma}(\boldsymbol{u}_S)\mathbf{n} = -\rho_F c^2\mathrm{div}\boldsymbol{u}_{F}\mathbf{n},
\end{align}

on the coupling boundary $\Gamma_I=\partial\Omega_F\cap\partial\Omega_F$. So, integrating over the domain $\Omega_F\cup\Omega_S$ and taking into account the symmetry of the tensor of elasticity, it holds

\begin{align}
-\omega^2\int_{\Omega_S} \rho_S\boldsymbol{u}_S\cdot \boldsymbol{v}_S\ \mathrm{d}\boldsymbol{x}
 + \int_{\Omega_S} \boldsymbol{\sigma}(\boldsymbol{u}_S) : \boldsymbol{\epsilon}(\boldsymbol{v}_S)\ \mathrm{d}\boldsymbol{x} 
 -\omega^2\int_{\Omega_F} \rho_F\boldsymbol{u}_F\cdot \boldsymbol{v}_F\ \mathrm{d}\boldsymbol{x}
 + \int_{\Omega_F} \rho_F c^2\mathrm{div}\boldsymbol{v}_F\,\mathrm{div}\boldsymbol{v}_F \mathrm{d}\boldsymbol{x} 
 = \int_{\Gamma_T} \boldsymbol{T}\cdot \boldsymbol{v}_S\ \mathrm{d}\boldsymbol{s}
\tag{8}
\end{align}

for all $(\boldsymbol{v}_F,\boldsymbol{v}_S)\in \hat{V}$ such that $\boldsymbol{u}_S=\mathbf{0}$ on the campled boundary $\Gamma_C=\{\mathbf{x}\in\partial\Omega_{S}: x_0=0\}$ and the traction boundary $\Gamma_T=\{\mathbf{x}\in\partial\Omega_{S}: x_0=L\}$. In addition,
$\boldsymbol{\epsilon}(\boldsymbol{v})$ is the symmetric part of $\boldsymbol{\nabla} \boldsymbol{v}$.

### Enforcing boundary conditions

Now let us consider how to enforce boundary conditions. 
For Dirichlet boundaries, we will enforce boundary-conditions strongly.
For these points, no test functions are associated with the Dirichlet nodes.

For traction boundary conditions, we will enforce the boundary condition
weakly using the variational form (8).
Similar to the Helmholtz equation, we require their corresponding test
functions $\boldsymbol{v}_S$ vanish along $\Gamma_C$.
Then, the boundary integral above has no effects for points on
$\partial\Omega_S\setminus\Gamma_T$.

### Summary of variational form
In summary, the variational problem is to find $\boldsymbol{u}$ in a vector function space $\hat{V}$ such that
\begin{equation}
a((\boldsymbol{u}_F,\boldsymbol{u}_S),(\boldsymbol{v}_F,\boldsymbol{v}_S)) = L((\boldsymbol{v}_F,\boldsymbol{v}_S))\quad\forall (\boldsymbol{v}_F,\boldsymbol{v}_S)\in\hat{V},
\end{equation}
where 

\begin{align}
a((\boldsymbol{u}_F,\boldsymbol{u}_S),(\boldsymbol{v}_F,\boldsymbol{v}_S)) &= -\omega^2\int_{\Omega_S} \rho_S\boldsymbol{u}_S\cdot \boldsymbol{v}_S\ \mathrm{d}\boldsymbol{x}
 + \int_{\Omega_S} \boldsymbol{\sigma}(\boldsymbol{u}_S) : \boldsymbol{\epsilon}(\boldsymbol{v}_S)\ \mathrm{d}\boldsymbol{x} 
 -\omega^2\int_{\Omega_F} \rho_F\boldsymbol{u}_F\cdot \boldsymbol{v}_F\ \mathrm{d}\boldsymbol{x}
 + \int_{\Omega_F} \rho_F c^2\mathrm{div}\boldsymbol{v}_F\,\mathrm{div}\boldsymbol{v}_F \mathrm{d}\boldsymbol{x} 
\end{align}
and 
\begin{equation}
\boldsymbol{\sigma}(\boldsymbol{u}_S) = \lambda(\boldsymbol{\nabla}\cdot \boldsymbol{u}_S)\boldsymbol{I} + \mu(\boldsymbol{\nabla} \boldsymbol{u}_S + (\boldsymbol{\nabla} \boldsymbol{u}_S)^{\top}).\\
\end{equation}

## NGSolve implementation

To demonstrate the implementation, we will model a clamped beam deformed under a time-harmonic surface force on the opposite free cross-section surface. This can be modeled by setting $\boldsymbol{T}=(0,0,1)$ on that boundary $\Gamma_T$. The solid structure is box-shaped with length and width $L$, whereas the fluid domain is an interior box-shaped domain with length and width $W<L$. We
set $\boldsymbol{u}=(0,0,0)$ at the clamped end, $x=0$. The rest of the lateral boundary is
traction free; that is, we set $\boldsymbol{T} = 0$. Therefore,
$$L((\boldsymbol{v}_F,\boldsymbol{v}_S)) = \int_{\Gamma_T} \boldsymbol{T}\cdot \boldsymbol{v}_S \mathrm{d}\boldsymbol{s}$$
for this problem.

### Import packages

We start by importing NGSolve.

In [18]:
import numpy as np
from ngsolve import *
from netgen.occ import *
from ngsolve.webgui import Draw

### Generate the mesh and function spaces

Our action starts by generating the mesh. In NGSolve we use Netgen for geometry and mesh generation. We define the outer solid domain and the inner fluid subdomain, assigning material names and boundary labels.

In [19]:
# Geometry
L = 1.0

# Primary geometric objects
domain_solid = Rectangle(L, L).Face()
domain_fluid = MoveTo(L/4, L/4).Rectangle(L/2, L/2).Face()

# Domain and boundary tags for the upper part
domain_solid.faces.name = "solid"
domain_solid.faces.col = (0, 1, 0)  # Green color for faces
domain_solid.mat("solid")
domain_solid.edges.Min(Y).name = "clamped"
domain_solid.edges.Min(Y).col = (1, 0, 0) # red
domain_solid.edges.Max(Y).name = "top"
domain_solid.edges.Max(Y).col = (0, 0, 1) # blue
domain_solid.edges.Min(X).name = "free"
domain_solid.edges.Min(X).col = (0, 0, 1) # blue
domain_solid.edges.Max(X).name = "free"
domain_solid.edges.Max(X).col = (0, 0, 1) # blue

# Domain and boundary tags for the lower part
domain_fluid.faces.name = "fluid"
domain_fluid.faces.col = (0, 0, 1)  # Blue color for faces
domain_fluid.mat("fluid")
domain_fluid.edges.name = "interface"
domain_fluid.edges.col = (1, 1, 0) # yellow

# Create the domain
domain = Glue([domain_solid, domain_fluid])

## Each edge is colored
Draw(domain, height="3vh")

WebGuiWidget(layout=Layout(height='3vh', width='100%'), value={'ngsolve_version': 'Netgen x.x', 'mesh_dim': 3,…

BaseWebGuiScene

### Visualise the subdomain markers

We colour each element by material (solid = 1, fluid = 2) to verify the partition.

In [20]:
# Define the mesh
h_size = 0.05
mesh = Mesh(OCCGeometry(domain, dim=2).GenerateMesh(maxh=h_size, quad_dominated=False))

print("Number of elements:", mesh.ne)
print("Materials:", mesh.GetMaterials())
print("Boundaries:", mesh.GetBoundaries())

# Plot mesh
Draw(mesh, height="3vh")

Number of elements: 938
Materials: ('solid', 'fluid')
Boundaries: ('clamped', 'free', 'free', 'top', 'interface', 'interface', 'interface', 'interface')


WebGuiWidget(layout=Layout(height='3vh', width='100%'), value={'gui_settings': {}, 'ngsolve_version': '6.2.260…

BaseWebGuiScene

### Define the variational problem

The primary unknown is the vector displacement field $\boldsymbol{u}$. We use a vector-valued $H^1$ space with quadratic (order 2) Lagrange elements, exactly as in the original FEniCS notebook.

In [21]:
# Individual spaces
V_s = VectorH1(mesh, order=2, dirichlet="clamped", complex=True, definedon="solid")  # displacement (solid)
V_f = HDiv(mesh, order=1, RT=True, complex=True, definedon="fluid")  # displacement (fluid)
V_p = Compress(H1(mesh, order=1, complex=True, definedon=mesh.Boundaries('interface')))   # pressure    (interface)

# Mixed (product) space: W = V_u x V_p
W = V_s * V_f * V_p
print("Total DOFs:", W.ndof)

Total DOFs: 5576


In [22]:
# Physical parameters
omega_val = 2 * np.pi * 1.0   # angular frequency
rho_fluid = 0.2               # fluid density
c_val     = 1.0               # speed of sound
rho       = 1.0               # solid density
beta      = 1.25
lambda_   = beta              # first Lamé parameter
mu_val    = 1.0               # second Lamé parameter

# Strain and stress tensors
def epsilon(u):
    return Sym(Grad(u))   # symmetric gradient

def sigma(u):
    return lambda_ * Trace(Grad(u)) * Id(2) + 2 * mu_val * epsilon(u)

# Trial and test pairs from the mixed space
(us, uf, p) = W.TrialFunction()
(vs, vf, q) = W.TestFunction()

# Normal vector on facets
n = specialcf.normal(mesh.dim)

# Domain-restricted differential measures
dx_solid = dx(definedon=mesh.Materials("solid"))
dx_fluid = dx(definedon=mesh.Materials("fluid"))
dI = ds(definedon=mesh.Boundaries("interface"))

# Bilinear form  a(u, v)
a = BilinearForm(W)
a += (-omega_val**2 * rho        * InnerProduct(us, vs)         ) * dx_solid
a += (InnerProduct(sigma(us), epsilon(vs))                       ) * dx_solid
a += (-omega_val**2 * rho_fluid  * InnerProduct(uf, vf)         ) * dx_fluid
a += (rho_fluid * c_val**2 * div(uf) * div(vf)                  ) * dx_fluid

# Fluid-structure coupling terms on the interface
# pressure loading on the solid test function
a += (-p * InnerProduct(vs, n)) * dI
# pressure loading on the fluid test function
a += (p * InnerProduct(vf.Trace(), n)) * dI

# Lagrange multiplier terms to enforce continuity of normal velocity across the interface
a += (InnerProduct(uf.Trace(), n) - InnerProduct(us, n)) * q * dI
a.Assemble()

# Linear form  L(v)  — traction (0, 1) on the right boundary
f_vec = CF((0, 1*np.exp(1j*np.pi/4)))  # Complex traction vector
L = LinearForm(W)
L += InnerProduct(f_vec, vs) * ds(definedon=mesh.Boundaries("top"))
L.Assemble()

print("Bilinear and linear forms assembled.")

Bilinear and linear forms assembled.


### Solve the variational problem

The Dirichlet condition $\boldsymbol{u}=(0,0)$ on the clamped boundary has already been declared in the space definition. We solve the resulting linear system directly.

In [23]:
# Compute solution
gfu = GridFunction(W)

# Apply the Dirichlet condition on the displacement component only
gfu.vec.data = a.mat.Inverse(W.FreeDofs()) * L.vec

# Extract displacement and pressure
us_sol, uf_sol, p_sol = gfu.components
print("System solved.")

System solved.


### Plot the solution

In [25]:
# First component of the solution (solid x-displacement)
Draw(us_sol[0], mesh.Materials("solid"), animate_complex=True, height="3vh", min=-0.075, max=0.075)
# Second component of the solution (solid y-displacement)
Draw(us_sol[1], mesh.Materials("solid"), animate_complex=True, height="3vh", min=-0.1, max=0.1)
# Total displacement magnitude
Draw(us_sol, mesh.Materials("solid"), deformation=0.5*us_sol.real, height="3vh", min=-0.075, max=0.075)
# First component of the solution (fluid x-displacement)
Draw(uf_sol[0], mesh.Materials("fluid"), animate_complex=True, height="3vh", min=-0.075, max=0.075)
# Second component of the solution (fluid y-displacement)
Draw(uf_sol[1], mesh.Materials("fluid"), animate_complex=True, height="3vh", min=-0.1, max=0.1)
# Total displacement magnitude
Draw(uf_sol, mesh.Materials("fluid"), deformation=0.5*uf_sol.real, height="3vh", min=-0.075, max=0.075)
# Pressure field in the fluid domain
Draw(rho_fluid * c_val**2 * div(uf_sol), mesh.Materials("fluid"), animate_complex=True, height="3vh", min=-0.1, max=0.1)

WebGuiWidget(layout=Layout(height='3vh', width='100%'), value={'gui_settings': {'Complex': {'phase': 0.0, 'spe…

WebGuiWidget(layout=Layout(height='3vh', width='100%'), value={'gui_settings': {'Complex': {'phase': 0.0, 'spe…

WebGuiWidget(layout=Layout(height='3vh', width='100%'), value={'gui_settings': {'Complex': {'phase': 0.0, 'spe…

WebGuiWidget(layout=Layout(height='3vh', width='100%'), value={'gui_settings': {'Complex': {'phase': 0.0, 'spe…

WebGuiWidget(layout=Layout(height='3vh', width='100%'), value={'gui_settings': {'Complex': {'phase': 0.0, 'spe…

WebGuiWidget(layout=Layout(height='3vh', width='100%'), value={'gui_settings': {'Complex': {'phase': 0.0, 'spe…

WebGuiWidget(layout=Layout(height='3vh', width='100%'), value={'gui_settings': {'Complex': {'phase': 0.0, 'spe…

BaseWebGuiScene